In [0]:
import time
import pandas as pd
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import upper
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, split
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from pyspark.sql.functions import min, max, count, from_json, current_timestamp, col, udf
from pyspark.sql.functions import row_number



In [0]:
spark.sql("""
    SELECT 
        type,
        COUNT(*) as event_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM github_archive.bronze.events
    GROUP BY type
    ORDER BY event_count DESC
""").show()

+--------------------+-----------+-----+
|                type|event_count|  pct|
+--------------------+-----------+-----+
|           PushEvent|   28750175|65.56|
|         CreateEvent|    4639965|10.58|
|    PullRequestEvent|    2676287| 6.10|
|          WatchEvent|    1780911| 4.06|
|   IssueCommentEvent|    1735574| 3.96|
|         DeleteEvent|    1080560| 2.46|
|PullRequestReview...|     899177| 2.05|
|         IssuesEvent|     702178| 1.60|
|PullRequestReview...|     539836| 1.23|
|           ForkEvent|     421784| 0.96|
|        ReleaseEvent|     201831| 0.46|
|         PublicEvent|     134746| 0.31|
|  CommitCommentEvent|     125628| 0.29|
|         MemberEvent|     111636| 0.25|
|         GollumEvent|      56232| 0.13|
+--------------------+-----------+-----+



### Event Types to Build Silver Tables For

In [0]:
%sql
-- Creating tables for relevant event types
-- CREATE TABLE IF NOT EXISTS github_archive.silver.push_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.create_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.pullrequest_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.watch_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.issuecomment_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.delete_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.issues_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.fork_event;
-- CREATE TABLE IF NOT EXISTS github_archive.silver.other_events;


### Design Your Silver Schema Namespace

In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS github_archive.silver.dim_actors; -- deduplicated actor master
-- CREATE TABLE IF NOT EXISTS github_archive.silver.dim_repos; -- deduplicated repository master
-- CREATE TABLE IF NOT EXISTS github_archive.silver._rejected_records; -- records that failed critical DQ
-- CREATE TABLE IF NOT EXISTS github_archive.silver._data_quality_log; -- DQ check results per run

### Understand What You Are Parsing and Where It Comes From

In [0]:
print("Actor Description:")
actor_desc = spark.sql("""
    DESCRIBE TABLE github_archive.bronze.events actor;
""").show()

actor_sample = spark.sql("""
    SELECT actor 
    FROM github_archive.bronze.events
    LIMIT 1 OFFSET 3;
""")

print(f"Actor example: {actor_sample.collect()[0][0]}")

print("\nRepo Description:")
repo_desc = spark.sql("""
    DESCRIBE TABLE github_archive.bronze.events repo;
""").show()

repo_sample = spark.sql("""
    SELECT repo 
    FROM github_archive.bronze.events
    LIMIT 1 OFFSET 3;
""")

print(f"Repo example: {repo_sample.collect()[0][0]}")

print("\nPayload Description:")
payload_desc = spark.sql("""
    DESCRIBE TABLE github_archive.bronze.events payload;
""").show()

payload_sample = spark.sql("""
    SELECT payload 
    FROM github_archive.bronze.events
    LIMIT 1 OFFSET 3;
""")

print(f"Payload example: {payload_sample.collect()[0][0]}")

Actor Description:
+---------+----------+
|info_name|info_value|
+---------+----------+
| col_name|     actor|
|data_type|    string|
|  comment|      NULL|
+---------+----------+

Actor example: {"id":155086666,"login":"lu146enza","display_login":"lu146enza","gravatar_id":"","url":"https://api.github.com/users/lu146enza","avatar_url":"https://avatars.githubusercontent.com/u/155086666?"}

Repo Description:
+---------+----------+
|info_name|info_value|
+---------+----------+
| col_name|      repo|
|data_type|    string|
|  comment|      NULL|
+---------+----------+

Repo example: {"id":739139151,"name":"lu146enza/Project1","url":"https://api.github.com/repos/lu146enza/Project1"}

Payload Description:
+---------+----------+
|info_name|info_value|
+---------+----------+
| col_name|   payload|
|data_type|    string|
|  comment|      NULL|
+---------+----------+

Payload example: {"repository_id":739139151,"push_id":16540296395,"size":1,"distinct_size":1,"ref":"refs/heads/main","head":"da88

### Build the Dimension Tables First

Building the actor dimension table

In [0]:
# Define the actor table dimensions
# Define a logic that parses for all the actor dimensions excluding the timestamps
# Define a window for partitioning the events


In [0]:
# spark.sql("DROP TABLE github_archive.silver.dim_actors")

# spark.sql("""
#     CREATE TABLE github_archive.silver.dim_actors (
#         actor_id            LONG,
#         actor_login         STRING,
#         actor_display_login STRING,
#         actor_url           STRING,
#         actor_avatar_url    STRING,
#         first_seen_at       TIMESTAMP,
#         last_seen_at        TIMESTAMP,
#         total_events        LONG,
#         silver_processed_at TIMESTAMP
#     )
# """)

DataFrame[]

In [0]:
# Define the expected structure
actor_schema = StructType([
    StructField("id", LongType()),
    StructField("login", StringType()),
    StructField("display_login", StringType()),
    StructField("url", StringType()),
    StructField("avatar_url", StringType())
])

# Apply from_json to convert the string into a struct
actors = spark.sql("SELECT actor, bronze_ingested_at FROM github_archive.bronze.events")
actors_transformed = actors.withColumn("actor_json", from_json(col("actor"), actor_schema))

# Extract individual fields from the struct using dot notation
# As a dataframe with struct objects, the column structs can be accessed through dot notations (like key value pairs)
actors_dim = actors_transformed.select(
    col("actor_json.id").alias("actor_id"),
    col("actor_json.login").alias("actor_login"),
    col("actor_json.display_login").alias("actor_display_login"),
    col("actor_json.url").alias("actor_url"),
    col("actor_json.avatar_url").alias("actor_avatar_url"),
    col("bronze_ingested_at").alias("ingestion_time")
)

# Aggregate all appearances per actor
actor_aggregates = actors_transformed.groupBy("actor_json.id").agg(
    min("bronze_ingested_at").alias("first_seen_at"),
    max("bronze_ingested_at").alias("last_seen_at"),
    count("*").alias("total_events")
)

# Deduplicate to get the most recent profile per actor
dedup_window = Window.partitionBy("actor_json.id").orderBy(col("bronze_ingested_at").desc())

actors_deduped = actors_transformed \
    .withColumn("rn", row_number().over(dedup_window)) \
    .filter(col("rn") == 1) \
    .select(
        col("actor_json.id").alias("actor_id"),
        col("actor_json.login").alias("actor_login"),
        col("actor_json.display_login").alias("actor_display_login"),
        col("actor_json.url").alias("actor_url"),
        col("actor_json.avatar_url").alias("actor_avatar_url"),
    )

# Join aggregates onto the deduplicated profiles
actors_dim = actors_deduped.join(
    actor_aggregates,
    actors_deduped["actor_id"] == actor_aggregates["id"],
    how="inner"
).drop("id") \
 .withColumn("silver_processed_at", current_timestamp())

display(actors_dim)
actors_dim.printSchema()
print(f"Total unique actors: {actors_dim.count()}")
# actors_dim.write.insertInto("github_archive.silver.dim_actors", overwrite=False)

actor_id,actor_login,actor_display_login,actor_url,actor_avatar_url,first_seen_at,last_seen_at,total_events,silver_processed_at
2851,matte,matte,https://api.github.com/users/matte,https://avatars.githubusercontent.com/u/2851?,2026-07-22T07:13:12.948Z,2026-07-22T07:13:12.948Z,1,2026-08-01T07:37:25.906Z
7312,rrees,rrees,https://api.github.com/users/rrees,https://avatars.githubusercontent.com/u/7312?,2026-07-21T07:40:05.772Z,2026-07-21T07:40:05.772Z,4,2026-08-01T07:37:25.906Z
7347,OptimisticMonkey,OptimisticMonkey,https://api.github.com/users/OptimisticMonkey,https://avatars.githubusercontent.com/u/7347?,2026-07-21T07:40:05.772Z,2026-07-21T07:40:05.772Z,3,2026-08-01T07:37:25.906Z
14640,bookest,bookest,https://api.github.com/users/bookest,https://avatars.githubusercontent.com/u/14640?,2026-07-21T07:40:04.296Z,2026-07-21T07:40:04.296Z,3,2026-08-01T07:37:25.906Z
15961,tomfast,tomfast,https://api.github.com/users/tomfast,https://avatars.githubusercontent.com/u/15961?,2026-07-21T07:40:05.772Z,2026-07-21T07:40:05.772Z,1,2026-08-01T07:37:25.906Z
31400,mertnuhoglu,mertnuhoglu,https://api.github.com/users/mertnuhoglu,https://avatars.githubusercontent.com/u/31400?,2026-07-21T07:40:05.772Z,2026-07-21T07:40:05.772Z,3,2026-08-01T07:37:25.906Z
36103,ChristianGutman,ChristianGutman,https://api.github.com/users/ChristianGutman,https://avatars.githubusercontent.com/u/36103?,2026-07-21T07:40:05.772Z,2026-07-21T07:40:05.772Z,2,2026-08-01T07:37:25.906Z
41730,abhinav,abhinav,https://api.github.com/users/abhinav,https://avatars.githubusercontent.com/u/41730?,2026-07-21T07:40:05.772Z,2026-07-22T07:13:12.948Z,39,2026-08-01T07:37:25.906Z
44649,fanf,fanf,https://api.github.com/users/fanf,https://avatars.githubusercontent.com/u/44649?,2026-07-21T07:40:05.772Z,2026-07-22T07:13:12.948Z,119,2026-08-01T07:37:25.906Z
52967,darjeeling,darjeeling,https://api.github.com/users/darjeeling,https://avatars.githubusercontent.com/u/52967?,2026-07-21T07:40:05.772Z,2026-07-21T07:40:05.772Z,1,2026-08-01T07:37:25.906Z


root
 |-- actor_id: long (nullable = true)
 |-- actor_login: string (nullable = true)
 |-- actor_display_login: string (nullable = true)
 |-- actor_url: string (nullable = true)
 |-- actor_avatar_url: string (nullable = true)
 |-- first_seen_at: timestamp (nullable = true)
 |-- last_seen_at: timestamp (nullable = true)
 |-- total_events: long (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)

Total unique actors: 2617527


In [0]:
actors_dim = actors_dim.withColumn("actor_id", col("actor_id").cast("long"))
actors_dim = actors_dim.filter(col("actor_id").isNotNull())
actors_dim.createOrReplaceTempView("actors_dim_staging")

# Verify the column is gone before running MERGE
actors_dim.printSchema()

root
 |-- actor_id: long (nullable = true)
 |-- actor_login: string (nullable = true)
 |-- actor_display_login: string (nullable = true)
 |-- actor_url: string (nullable = true)
 |-- actor_avatar_url: string (nullable = true)
 |-- first_seen_at: timestamp (nullable = true)
 |-- last_seen_at: timestamp (nullable = true)
 |-- total_events: long (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
# Write using MERGE INTO

spark.sql("""
    MERGE INTO github_archive.silver.dim_actors AS target
    USING actors_dim_staging AS source
    ON target.actor_id = CAST(source.actor_id AS LONG)
    WHEN MATCHED THEN UPDATE SET
        target.actor_login          = source.actor_login,
        target.actor_display_login  = source.actor_display_login,
        target.actor_url            = source.actor_url,
        target.actor_avatar_url     = source.actor_avatar_url,
        target.last_seen_at         = source.last_seen_at,
        target.total_events         = source.total_events,
        target.silver_processed_at  = source.silver_processed_at
    WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Check source type
actors_dim.printSchema()

# Check target type
spark.sql("DESCRIBE github_archive.silver.dim_actors").show()

root
 |-- actor_id: long (nullable = true)
 |-- actor_login: string (nullable = true)
 |-- actor_display_login: string (nullable = true)
 |-- actor_url: string (nullable = true)
 |-- actor_avatar_url: string (nullable = true)
 |-- first_seen_at: timestamp (nullable = true)
 |-- last_seen_at: timestamp (nullable = true)
 |-- total_events: long (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)

+-------------------+---------+-------+
|           col_name|data_type|comment|
+-------------------+---------+-------+
|           actor_id|   bigint|   NULL|
|        actor_login|   string|   NULL|
|actor_display_login|   string|   NULL|
|          actor_url|   string|   NULL|
|   actor_avatar_url|   string|   NULL|
|      first_seen_at|timestamp|   NULL|
|       last_seen_at|timestamp|   NULL|
|       total_events|   bigint|   NULL|
|silver_processed_at|timestamp|   NULL|
+-------------------+---------+-------+



In [0]:
spark.sql("""
    SELECT 
        SUM(total_events)   AS total_events_sum,
        COUNT(*)            AS unique_actors
    FROM github_archive.silver.dim_actors
""").show()

+----------------+-------------+
|total_events_sum|unique_actors|
+----------------+-------------+
|        43856454|      2617527|
+----------------+-------------+



In [0]:
spark.sql("SELECT COUNT(*) as count FROM github_archive.silver.dim_actors").show()

+-------+
|  count|
+-------+
|2617527|
+-------+



In [0]:
spark.sql("""
    SELECT COUNT(*) AS bronze_row_count
    FROM github_archive.bronze.events
""").show()

+----------------+
|bronze_row_count|
+----------------+
|        43856454|
+----------------+



Building the repo dimension table

In [0]:
%sql
-- -- Declaring the columns of dim_repos
-- ALTER TABLE github_archive.silver.dim_repos 
-- ADD COLUMNS (
--     repo_id LONG,
--     repo_name STRING,
--     repo_owner STRING,
--     repo_short_name STRING,
--     repo_url STRING,
--     first_seen_at STRING,
--     last_seen_at STRING,
--     total_events LONG,
--     silver_processed_at TIMESTAMP
-- )


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4702335465471409>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '-- Declaring the columns of dim_repos\nALTER TABLE github_archive.silver.dim_repos \nADD COLUMNS (\n    repo_id LONG,\n    repo_name STRING,\n    repo_owner STRING,\n    repo_short_name STRING,\n    repo_url STRING,\n    first_seen_at STRING,\n    last_seen_at STRING,\n    total_events LONG,\n    silver_processed_at TIMESTAMP\n)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the la

In [0]:
# Define the expected structure

# Repo example: {"id":739139151,"name":"lu146enza/Project1","url":"https://api.github.com/repos/lu146enza/Project1"}
repo_schema = StructType([
    StructField("id", LongType()),
    StructField("name", StringType()),
    StructField("url", StringType())
])

# Apply from_json to convert the string into a struct
repos = spark.sql("SELECT repo, bronze_ingested_at FROM github_archive.bronze.events")
repos_transformed = repos.withColumn("repos_json", from_json(col("repo"), repo_schema))

# Extract individual fields from the struct using dot notation
# As a dataframe with struct objects, the column structs can be accessed through dot notations (like key value pairs)
repos_dim = repos_transformed.select(
    col("repos_json.id").alias("repo_id"),
    col("repos_json.name").alias("repo_name"),
    split(col("repos_json.name"), "/")[0].alias("repo_owner"),
    split(col("repos_json.name"), "/")[1].alias("repo_short_name"),
    col("repos_json.url").alias("repo_url")
)

# display(repos_dim)
repos_dim.printSchema()
print(f"Total unique repos: {repos_dim.count()}")
# actors_dim.write.insertInto("github_archive.silver.dim_actors", overwrite=False)

root
 |-- repo_id: long (nullable = true)
 |-- repo_name: string (nullable = true)
 |-- repo_owner: string (nullable = true)
 |-- repo_short_name: string (nullable = true)
 |-- repo_url: string (nullable = true)

Total unique repos: 210676185


In [0]:
# Aggregate all appearances per actor
repos_aggregates = repos_transformed.groupBy("repos_json.id").agg(
    min("bronze_ingested_at").alias("first_seen_at"),
    max("bronze_ingested_at").alias("last_seen_at"),
    count("*").alias("total_events")
)

# Deduplicate to get the most recent profile per actor
dedup_window = Window.partitionBy("repos_json.id").orderBy(col("bronze_ingested_at").desc())

repos_deduped = repos_transformed \
    .withColumn("rn", row_number().over(dedup_window)) \
    .filter(col("rn") == 1) \
    .select(
        col("repos_json.id").alias("repo_id"),
        col("repos_json.name").alias("repo_name"),
        split(col("repos_json.name"), "/")[0].alias("repo_owner"),
        split(col("repos_json.name"), "/")[1].alias("repo_short_name"),
        col("repos_json.url").alias("repo_url")
    )

# Join aggregates onto the deduplicated profiles
repos_dim = repos_deduped.join(
    repos_aggregates,
    repos_deduped["repo_id"] == repos_aggregates["id"],
    how="inner"
).drop("id") \
 .withColumn("silver_processed_at", current_timestamp())

# repos_transformed.unpersist()

# display(repos_dim)
repos_dim.printSchema()
print(f"Total unique repos: {repos_dim.count()}")
# repos_dim.write.insertInto("github_archive.silver.dim_repos", overwrite=False)

root
 |-- repo_id: long (nullable = true)
 |-- repo_name: string (nullable = true)
 |-- repo_owner: string (nullable = true)
 |-- repo_short_name: string (nullable = true)
 |-- repo_url: string (nullable = true)
 |-- first_seen_at: timestamp (nullable = true)
 |-- last_seen_at: timestamp (nullable = true)
 |-- total_events: long (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)

Total unique repos: 13232122


In [0]:
repos_dim.filter(col("repo_id").isNull()).count()

0

In [0]:
%sql

DESCRIBE github_archive.silver.dim_repos

col_name,data_type,comment
repo_id,bigint,null
repo_name,string,null
repo_owner,string,null
repo_short_name,string,null
repo_url,string,null
first_seen_at,string,null
last_seen_at,string,null
total_events,bigint,null
silver_processed_at,timestamp,null


In [0]:
%sql
TRUNCATE TABLE github_archive.silver.dim_repos

In [0]:
# Then register the view — not before
repos_dim.createOrReplaceTempView("repos_dim_staging")

# Verify the column is gone before running MERGE
repos_dim.printSchema()

root
 |-- repo_id: long (nullable = true)
 |-- repo_name: string (nullable = true)
 |-- repo_owner: string (nullable = true)
 |-- repo_short_name: string (nullable = true)
 |-- repo_url: string (nullable = true)
 |-- first_seen_at: timestamp (nullable = true)
 |-- last_seen_at: timestamp (nullable = true)
 |-- total_events: long (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
# Write using MERGE INTO

# repos_dim.createOrReplaceTempView("repos_dim_staging")

spark.sql("""
    MERGE INTO github_archive.silver.dim_repos AS target
    USING repos_dim_staging AS source
    ON target.repo_id = source.repo_id
    WHEN MATCHED THEN UPDATE SET
        target.repo_name          = source.repo_name,
        target.repo_owner  = source.repo_owner,
        target.repo_short_name            = source.repo_short_name,
        target.repo_url     = source.repo_url,
        target.first_seen_at     = source.first_seen_at,
        target.last_seen_at         = source.last_seen_at,
        target.total_events         = source.total_events,
        target.silver_processed_at  = source.silver_processed_at
    WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
    SELECT 
        SUM(total_events)   AS total_events_sum,
        COUNT(*)            AS unique_repos
    FROM github_archive.silver.dim_repos
""").show()

+----------------+------------+
|total_events_sum|unique_repos|
+----------------+------------+
|       210676185|    13232122|
+----------------+------------+



Understand the Payload Schema Definition Pattern

In [0]:
# Sample 10 payload values for PushEvent
spark.sql("""
    SELECT payload 
    FROM github_archive.bronze.events
    WHERE type = 'PushEvent'
    LIMIT 10
""").show(truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
spark.sql("""
    SELECT 
        (SELECT COUNT(*) FROM github_archive.silver.push_events)            +
        (SELECT COUNT(*) FROM github_archive.silver.pullrequest_event)    +
        (SELECT COUNT(*) FROM github_archive.silver.issues_event)           +
        (SELECT COUNT(*) FROM github_archive.silver.watch_event)           +
        (SELECT COUNT(*) FROM github_archive.silver.fork_event)            +
        (SELECT COUNT(*) FROM github_archive.silver.create_event)          +
        (SELECT COUNT(*) FROM github_archive.silver.delete_event)          +
        (SELECT COUNT(*) FROM github_archive.silver.other_events)           
        AS total_silver_rows
""").show()

+-----------------+
|total_silver_rows|
+-----------------+
|         43856454|
+-----------------+



In [0]:
# bronze = 43856454
# silver = 43856454
# try:
#     rejected = spark.sql("SELECT COUNT(*) FROM github_archive.silver._rejected_records").collect()[0][0]
# except Exception:
#     rejected = 0

# accounted = silver + rejected
# unaccounted = bronze - accounted

# print(f"Bronze rows:      {bronze:,}")
# print(f"Silver rows:      {silver:,}")
# print(f"Rejected rows:    {rejected:,}")
# print(f"Accounted for:    {accounted:,}")
# print(f"Unaccounted:      {unaccounted:,}")
# print(f"Status:           {'PASS' if unaccounted == 0 else 'FAIL'}")

Bronze rows:      43,856,520
Silver rows:      43,856,454
Rejected rows:    0
Accounted for:    43,856,454
Unaccounted:      66
Status:           FAIL


In [0]:
# spark.sql("""
#     SELECT type, COUNT(*) as bronze_count
#     FROM github_archive.bronze.events
#     GROUP BY type
#     ORDER BY bronze_count DESC
# """).show(20)

+--------------------+------------+
|                type|bronze_count|
+--------------------+------------+
|           PushEvent|    28750171|
|         CreateEvent|     4639963|
|    PullRequestEvent|     2676271|
|          WatchEvent|     1780905|
|   IssueCommentEvent|     1735562|
|         DeleteEvent|     1080556|
|PullRequestReview...|      899173|
|         IssuesEvent|      702176|
|PullRequestReview...|      539836|
|           ForkEvent|      421768|
|        ReleaseEvent|      201831|
|         PublicEvent|      134746|
|  CommitCommentEvent|      125628|
|         MemberEvent|      111636|
|         GollumEvent|       56232|
+--------------------+------------+



In [0]:
# silver_counts = {
#     "PushEvent":            spark.sql("SELECT COUNT(*) FROM github_archive.silver.push_events").collect()[0][0],
#     "PullRequestEvent":     spark.sql("SELECT COUNT(*) FROM github_archive.silver.pullrequest_event").collect()[0][0],
#     "IssuesEvent":          spark.sql("SELECT COUNT(*) FROM github_archive.silver.issues_event").collect()[0][0],
#     "WatchEvent":           spark.sql("SELECT COUNT(*) FROM github_archive.silver.watch_event").collect()[0][0],
#     "ForkEvent":            spark.sql("SELECT COUNT(*) FROM github_archive.silver.fork_event").collect()[0][0],
#     "CreateEvent":          spark.sql("SELECT COUNT(*) FROM github_archive.silver.create_event").collect()[0][0],
#     "DeleteEvent":          spark.sql("SELECT COUNT(*) FROM github_archive.silver.delete_event").collect()[0][0],
# }

# # Get Bronze counts per handled event type
# bronze_counts = spark.sql("""
#     SELECT type, COUNT(*) as count
#     FROM github_archive.bronze.events
#     WHERE type IN ('PushEvent','PullRequestEvent','IssuesEvent',
#                    'WatchEvent','ForkEvent','CreateEvent','DeleteEvent')
#     GROUP BY type
# """).collect()

# print(f"{'Event Type':<30} {'Bronze':>10} {'Silver':>10} {'Gap':>10}")
# print("-" * 62)
# for row in bronze_counts:
#     event_type = row['type']
#     bronze = row['count']
#     silver = silver_counts.get(event_type, 0)
#     gap = bronze - silver
#     print(f"{event_type:<30} {bronze:>10,} {silver:>10,} {gap:>10,}")

Event Type                         Bronze     Silver        Gap
--------------------------------------------------------------
ForkEvent                         421,768    421,768          0
DeleteEvent                     1,080,556  1,080,556          0
PullRequestEvent                2,676,271  2,676,271          0
WatchEvent                      1,780,905  1,780,905          0
CreateEvent                     4,639,963  4,639,963          0
IssuesEvent                       702,176    702,176          0
PushEvent                      28,750,171 28,750,171          0


In [0]:
# spark.sql("""
#     SELECT 
#         (SELECT COUNT(*) FROM github_archive.bronze.events 
#          WHERE type NOT IN ('PushEvent','PullRequestEvent','IssuesEvent',
#                             'WatchEvent','ForkEvent','CreateEvent','DeleteEvent')) 
#         AS bronze_other,
        
#         (SELECT COUNT(*) FROM github_archive.silver.other_events) 
#         AS silver_other
# """).show()

+------------+------------+
|bronze_other|silver_other|
+------------+------------+
|     3804644|     3804644|
+------------+------------+



In [0]:
# bronze = spark.sql("SELECT COUNT(*) FROM github_archive.bronze.events").collect()[0][0]
# silver = spark.sql("""
#     SELECT 
#         (SELECT COUNT(*) FROM github_archive.silver.push_events)            +
#         (SELECT COUNT(*) FROM github_archive.silver.pullrequest_event)    +
#         (SELECT COUNT(*) FROM github_archive.silver.issues_event)           +
#         (SELECT COUNT(*) FROM github_archive.silver.watch_event)           +
#         (SELECT COUNT(*) FROM github_archive.silver.fork_event)            +
#         (SELECT COUNT(*) FROM github_archive.silver.create_event)          +
#         (SELECT COUNT(*) FROM github_archive.silver.delete_event)          +
#         (SELECT COUNT(*) FROM github_archive.silver.other_events)           
#         AS total_silver_rows
# """).collect()[0][0]

# print(f"Bronze: {bronze:,}")
# print(f"Silver: {silver:,}")
# print(f"Gap:    {bronze - silver:,}")

Bronze: 43,856,454
Silver: 43,856,454
Gap:    0


In [0]:
# spark.sql("""
#     DESCRIBE HISTORY github_archive.bronze.events
# """).select("version", "timestamp", "operation", "operationParameters").show(10, truncate=False)

+-------+-------------------+-----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation        |operationParameters                                                                                                                                                                                                                                                                                                                                                         